# PCB Isolation Design Walkthrough: IEC 60664-1 Clearance & Creepage

This notebook works through a real clearance/creepage sizing problem for a
motor-drive PCB, following the method described in Texas Instruments'
application note *"Circuit Board Insulation Design According to IEC60664 for
Motor Drive Application"* (SLUAAR5, August 2023).

Rather than just restating the numbers from the white paper, this notebook
**rebuilds the calculation in Python**, using small classes to represent:

- the **operating environment** (pollution degree, material group, altitude)
- the **voltage domains** on the board (mains input, DC bus, control logic,
  earth, isolated/user side, motor output)
- the **insulation barrier** between any two voltage domains (insulation
  type, overvoltage category, working voltage)
- a **calculator** that turns those inputs into required clearance and
  creepage distances, and finally into PCB design-rule spacings

> Safety note: this notebook is a *teaching example*, not a certified
> design tool. The lookup tables below only contain the handful of reference
> points that the source white paper works out explicitly. A real product
> must pull the complete tables from IEC 60664-1 (and, for drives, IEC
> 61800-5-1) and be reviewed by a qualified safety engineer.

## 1. Two different failure modes, two different distances

IEC 60664-1 asks for two independent spacings between any two conductors at
different potentials:

- **Clearance** - the shortest straight-line distance *through air*. It
  protects against a fast breakdown (arcing) caused by voltage spikes and
  transients, so it's sized from the **peak/impulse voltage**, including the
  overvoltage category (OVC).
- **Creepage** - the shortest path *along an insulating surface* between the
  same two conductors. It protects against a slow failure mode (surface
  tracking) caused by dirt, humidity and contamination over time, so it's
  sized from the **RMS working voltage**, the pollution degree, and the
  insulating material's comparative tracking index (its "material group").

Because creepage is measured along a surface between the same two points,
it's always geometrically at least as long as clearance - but the two
values are still calculated independently, and a real layout has to respect
both.

The insulation itself is also classified by *how much protection* it has to
provide:

| Type | Purpose |
|---|---|
| Functional | lets the circuit work; no shock protection |
| Basic | single layer of shock protection |
| Reinforced | equivalent to two independent basic layers, in a single barrier |

## 2. The example system

We'll use the same worked example as the white paper: an industrial AC motor
drive.

- Class I equipment, metal cabinet, chassis bonded to protective earth (PE)
- 3-phase, 220/380 V wye mains supply, overvoltage category III
- Industrial pollution degree 2 environment
- Galvanically isolated HMI (human-machine interface) required
- "Hot-side" MCU - the microcontroller and gate-drive circuitry sit on the
  same (non-isolated) potential as the power stage
- Installation altitude below 2000 m

The power stage is a standard 3-phase rectifier to DC bus to 3-phase
inverter, with the MCU/gate-driver circuitry referenced to the power stage's
local ground, and a small isolated supply feeding a user-facing HMI on the
"cold" side. We'll group the board into five logical **voltage domains**,
matching the block labels A-W used in the white paper's schematic:

| Domain | Blocks | What it is |
|---|---|---|
| `MAINS` | A, B, C | 3-phase 380 Vac line input, upstream of the rectifier |
| `DC_BUS` | D, E, F | Rectified DC bus and hot-side control/gate-drive circuitry |
| `EARTH` | G | Protective earth (PE) |
| `COLD` | H | Isolated ("cold"/user) side reference, after the isolated aux supply |
| `MOTOR` | U, V, W | Inverter output phases to the motor |

Every pair of domains needs its own clearance/creepage spec, because both
the insulation type (functional/basic/reinforced) and the working voltage
across a barrier depend on which two domains you're looking at.

In [25]:
import math
from dataclasses import dataclass
from enum import Enum

import pandas as pd

pd.set_option("display.width", 100)

## 3. Modeling the environment

Pollution degree, PCB material group (CTI), and installation altitude all
shift the required distances. We capture them once, in one place, as an
`Environment` object, so every calculation downstream stays consistent.

Altitude matters because clearance is really about the dielectric strength
of *air*, and air gets thinner (and a worse insulator) at altitude - above
2000 m, IEC 60664-1 requires clearance to be multiplied by a correction
factor from Table A.2.

In [26]:
class PollutionDegree(Enum):
    PD1 = 1
    PD2 = 2
    PD3 = 3
    PD4 = 4


@dataclass
class Environment:
    pollution_degree: PollutionDegree
    material_group: str = "IIIa"   # PCB laminate CTI group, e.g. IIIa for standard FR-4
    altitude_m: float = 2000

    # Altitude correction factor, IEC 60664-1 Table A.2 (only the two points
    # explicitly used in the source white paper are included here).
    _ALTITUDE_CORRECTION = {2000: 1.00, 4000: 1.29}

    def altitude_correction_factor(self) -> float:
        try:
            return self._ALTITUDE_CORRECTION[self.altitude_m]
        except KeyError:
            raise ValueError(
                f"No altitude-correction reference point at {self.altitude_m} m in this "
                "notebook's excerpted table -- look up IEC 60664-1 Table A.2 directly."
            )

    def describe(self) -> str:
        return (
            f"Pollution degree {self.pollution_degree.value}, "
            f"material group {self.material_group}, "
            f"altitude {self.altitude_m} m "
            f"(clearance altitude factor x{self.altitude_correction_factor():.2f})"
        )


env = Environment(pollution_degree=PollutionDegree.PD2, material_group="IIIa", altitude_m=2000)
print(env.describe())

Pollution degree 2, material group IIIa, altitude 2000 m (clearance altitude factor x1.00)


## 4. Modeling the voltage domains

Each `VoltageBlock` is just a labeled node on the board. We don't attach a
working voltage to the block itself, because the *relevant* working voltage
between two domains depends on which pair you're evaluating - mains-to-earth
sees a different working voltage than mains-to-motor-output, for example.

In [27]:
@dataclass
class VoltageBlock:
    name: str
    description: str


MAINS = VoltageBlock("MAINS", "3x380 Vac wye mains input (A/B/C phases)")
DC_BUS = VoltageBlock("DC_BUS", "Rectified DC bus + hot-side MCU/gate-drive circuitry (D/E/F)")
EARTH = VoltageBlock("EARTH", "Protective earth / chassis (G)")
COLD = VoltageBlock("COLD", "Isolated 'cold'/user-side reference, after the aux supply (H)")
MOTOR = VoltageBlock("MOTOR", "Inverter output phases to the motor (U/V/W)")

domains = [MAINS, DC_BUS, EARTH, COLD, MOTOR]
for d in domains:
    print(f"{d.name:8s} - {d.description}")

MAINS    - 3x380 Vac wye mains input (A/B/C phases)
DC_BUS   - Rectified DC bus + hot-side MCU/gate-drive circuitry (D/E/F)
EARTH    - Protective earth / chassis (G)
COLD     - Isolated 'cold'/user-side reference, after the aux supply (H)
MOTOR    - Inverter output phases to the motor (U/V/W)


## 5. Modeling a barrier between two domains

An `InsulationBarrier` is the actual thing we need a spacing for: a specific
pair of voltage domains, the insulation type required between them, the
overvoltage category, the impulse (peak) voltage used for clearance, and the
RMS working voltage used for creepage.

The impulse and working voltages below are taken from the white paper's own
Step 3 / Step 4 analysis (its Tables 2-1 and 2-2) - in a real design these
come from simulating or measuring the actual circuit, not from a lookup
table.

In [28]:
class InsulationType(Enum):
    FUNCTIONAL = "functional"
    BASIC = "basic"
    REINFORCED = "reinforced"


class OvervoltageCategory(Enum):
    OVC_I = "I"
    OVC_II = "II"
    OVC_III = "III"
    OVC_IV = "IV"


@dataclass
class InsulationBarrier:
    domain_a: VoltageBlock
    domain_b: VoltageBlock
    insulation_type: InsulationType
    ovc: OvervoltageCategory
    impulse_voltage_v: float       # peak, drives the clearance lookup
    working_voltage_rms_v: float   # rms, drives the creepage lookup

    @property
    def label(self) -> str:
        return f"{self.domain_a.name} - {self.domain_b.name}"

    def __repr__(self):
        return f"<Barrier {self.label}: {self.insulation_type.value}, OVC {self.ovc.value}>"


## 6. The clearance/creepage calculator

This is the engineering core of the notebook. It implements the two
formulas the white paper works out by hand:

**Clearance** is read straight off a table of impulse voltage to clearance,
for a given pollution degree (Table F.2 of IEC 60664-1). The white paper
works out three specific points for this system (220/380 V mains, pollution
degree 2):

| Insulation | OVC | Impulse voltage | Clearance |
|---|---|---|---|
| Functional | I | 1500 V | 0.5 mm |
| Basic | III | 4000 V | 3.0 mm |
| Reinforced | III | 6000 V | 5.5 mm |

**Creepage** is read off a table of working voltage (rms) to creepage, for a
given pollution degree and material group (Table F.4), and it's the *rms*
working voltage, not the peak - creepage failure is a slow process governed
by average voltage stress. Table F.4 only tabulates a handful of standard
voltage steps, so intermediate voltages are found by **linear interpolation
between the two bracketing steps**. The white paper gives three such points:

| Working voltage | Creepage (functional/basic) |
|---|---|
| 400 V | 2.00 mm |
| 500 V | 2.50 mm |
| 630 V | 3.20 mm |

Reinforced insulation gets **double** the basic/functional creepage value at
the same working voltage.

We encode exactly these reference points below - nothing more. If you ask
the calculator for a voltage it doesn't have a bracketing pair for, it
raises an error rather than silently guessing, since fabricating an
insulation-safety number is worse than admitting you don't have it.

In [29]:
class IEC60664Calculator:
    """
    Clearance/creepage calculator built from the specific worked example in
    TI's SLUAAR5 application note, for Pollution Degree 2 / material group
    IIIa. This intentionally is NOT a complete implementation of IEC 60664-1
    Annex F -- see the markdown cell above for exactly which reference
    points it knows. Extend _CLEARANCE_POINTS / _CREEPAGE_POINTS with the
    full tables from the standard before using this for a real design.
    """

    _CLEARANCE_POINTS = {   # impulse voltage (V, peak) -> clearance (mm), PD2
        1500: 0.5,
        4000: 3.0,
        6000: 5.5,
    }

    _CREEPAGE_POINTS = {    # working voltage (V, rms) -> basic/functional creepage (mm), PD2/IIIa
        400: 2.00,
        500: 2.50,
        630: 3.20,
    }

    def __init__(self, environment):
        self.env = environment

    def clearance_mm(self, impulse_voltage_v: float) -> float:
        if impulse_voltage_v not in self._CLEARANCE_POINTS:
            raise ValueError(
                f"No reference point for a {impulse_voltage_v} V impulse in this notebook's "
                "excerpted table -- look up IEC 60664-1 Table F.2 directly."
            )
        return self._CLEARANCE_POINTS[impulse_voltage_v]

    def creepage_mm(self, working_voltage_rms_v: float, insulation_type: InsulationType) -> float:
        points = sorted(self._CREEPAGE_POINTS.items())
        lower = max((p for p in points if p[0] <= working_voltage_rms_v), default=None)
        upper = min((p for p in points if p[0] >= working_voltage_rms_v), default=None)
        if lower is None or upper is None:
            raise ValueError(
                f"{working_voltage_rms_v} V rms falls outside the "
                f"{points[0][0]}-{points[-1][0]} V range transcribed from the white paper -- "
                "look up IEC 60664-1 Table F.4 directly."
            )
        if lower == upper:
            basic = lower[1]
        else:
            v0, c0 = lower
            v1, c1 = upper
            basic = (working_voltage_rms_v - v0) * (c1 - c0) / (v1 - v0) + c0
        return 2 * basic if insulation_type == InsulationType.REINFORCED else basic

    def evaluate(self, barrier: InsulationBarrier) -> dict:
        clearance = round(self.clearance_mm(barrier.impulse_voltage_v), 2)
        creepage = round(self.creepage_mm(barrier.working_voltage_rms_v, barrier.insulation_type), 2)
        return {
            "barrier": barrier.label,
            "insulation": barrier.insulation_type.value,
            "OVC": barrier.ovc.value,
            "impulse_V": barrier.impulse_voltage_v,
            "working_V_rms": barrier.working_voltage_rms_v,
            "clearance_mm": clearance,
            "creepage_mm": creepage,
        }

    def altitude_corrected_clearance_mm(self, clearance_mm: float) -> float:
        return round(clearance_mm * self.env.altitude_correction_factor(), 2)


calc = IEC60664Calculator(env)

## 7. Reproducing the white paper's worked numbers

Before applying the calculator to the whole board, let's check it against
the exact numbers the white paper derives by hand.

In [30]:
print("Clearance (Table F.2, PD2):")
for label, iv in [("functional, OVC I", 1500), ("basic, OVC III", 4000), ("reinforced, OVC III", 6000)]:
    print(f"  {label:22s} {iv:5d} V impulse -> {calc.clearance_mm(iv)} mm")

print()
print("Creepage (Table F.4, PD2 / IIIa):")
for wv in (400, 440, 565):
    val = round(calc.creepage_mm(wv, InsulationType.BASIC), 2)
    print(f"  {wv} V rms, basic/functional -> {val} mm")
val = round(calc.creepage_mm(400, InsulationType.REINFORCED), 2)
print(f"  400 V rms, reinforced         -> {val} mm")

Clearance (Table F.2, PD2):
  functional, OVC I       1500 V impulse -> 0.5 mm
  basic, OVC III          4000 V impulse -> 3.0 mm
  reinforced, OVC III     6000 V impulse -> 5.5 mm

Creepage (Table F.4, PD2 / IIIa):
  400 V rms, basic/functional -> 2.0 mm
  440 V rms, basic/functional -> 2.2 mm
  565 V rms, basic/functional -> 2.85 mm
  400 V rms, reinforced         -> 4.0 mm


## 8. Applying it across the whole board

Now we define one `InsulationBarrier` per pair of domains, using the
insulation type / OVC / working-voltage assignments the white paper works
out in its Step 3 and Step 4 (Tables 2-1 and 2-2), and run every barrier
through the calculator.

In [31]:
barriers = [
    InsulationBarrier(MAINS,  MAINS,  InsulationType.BASIC,      OvervoltageCategory.OVC_III, 4000, 400),
    InsulationBarrier(MAINS,  DC_BUS, InsulationType.BASIC,      OvervoltageCategory.OVC_III, 4000, 400),
    InsulationBarrier(MAINS,  EARTH,  InsulationType.BASIC,      OvervoltageCategory.OVC_III, 4000, 400),
    InsulationBarrier(MAINS,  COLD,   InsulationType.REINFORCED, OvervoltageCategory.OVC_III, 6000, 440),
    InsulationBarrier(MAINS,  MOTOR,  InsulationType.BASIC,      OvervoltageCategory.OVC_III, 4000, 440),
    InsulationBarrier(DC_BUS, DC_BUS, InsulationType.FUNCTIONAL, OvervoltageCategory.OVC_I,   1500, 565),
    InsulationBarrier(DC_BUS, EARTH,  InsulationType.BASIC,      OvervoltageCategory.OVC_III, 4000, 400),
    InsulationBarrier(DC_BUS, COLD,   InsulationType.REINFORCED, OvervoltageCategory.OVC_III, 6000, 440),
    InsulationBarrier(DC_BUS, MOTOR,  InsulationType.FUNCTIONAL, OvervoltageCategory.OVC_I,   1500, 565),
    InsulationBarrier(EARTH,  MOTOR,  InsulationType.BASIC,      OvervoltageCategory.OVC_III, 4000, 400),
    InsulationBarrier(COLD,   MOTOR,  InsulationType.REINFORCED, OvervoltageCategory.OVC_III, 6000, 440),
    InsulationBarrier(MOTOR,  MOTOR,  InsulationType.FUNCTIONAL, OvervoltageCategory.OVC_I,   1500, 400),
]

results = pd.DataFrame([calc.evaluate(b) for b in barriers])
results

,barrier,insulation,OVC,impulse_V,working_V_rms,clearance_mm,creepage_mm
0,MAINS - MAINS,basic,III,4000,400,3.0,2.00
1,MAINS - DC_BUS,basic,III,4000,400,3.0,2.00
2,MAINS - EARTH,basic,III,4000,400,3.0,2.00
3,MAINS - COLD,reinforced,III,6000,440,5.5,4.40
4,MAINS - MOTOR,basic,III,4000,440,3.0,2.20
5,DC_BUS - DC_BUS,functional,I,1500,565,0.5,2.85
6,DC_BUS - EARTH,basic,III,4000,400,3.0,2.00
7,DC_BUS - COLD,reinforced,III,6000,440,5.5,4.40
8,DC_BUS - MOTOR,functional,I,1500,565,0.5,2.85
9,EARTH - MOTOR,basic,III,4000,400,3.0,2.00


## 9. Checking against the white paper, and one thing to flag

Comparing the table above to Table 2-3 of the white paper: every clearance
value matches exactly, and every creepage value matches **except** the
three `reinforced` barriers (`MAINS-COLD`, `DC_BUS-COLD`, `COLD-MOTOR`). The
formula (double the basic/functional creepage at 440 V, i.e. 2 x 2.2 mm)
gives **4.4 mm**, while the white paper's Table 2-3 lists **4.0 mm** for
those same barriers.

That's a small, real discrepancy in the source document -- most likely
because whoever produced the table doubled the creepage at the *nearest
round reference voltage* (400 V -> 2.0 mm -> 4.0 mm) rather than the
interpolated 440 V value. It doesn't change the overall design (4.4 mm is
the more conservative number), but it's a good reminder to **re-derive**
safety-relevant numbers from the underlying formulas rather than trusting a
table cell at face value -- including the ones this notebook produces.

One barrier is missing from the table above: `EARTH-COLD` (the G-H pair in
the original schematic), a low-voltage control-reference connection at
roughly 50 V (clearance) / 10 V (creepage). That's well below the 400-630 V
window this notebook's excerpted tables cover, so instead of guessing we
take the white paper's published value directly and label it accordingly.

In [32]:
given_values = pd.DataFrame([{
    "barrier": "EARTH - COLD",
    "insulation": "basic",
    "OVC": "I",
    "impulse_V": None,
    "working_V_rms": 10,
    "clearance_mm": 0.2,
    "creepage_mm": 0.04,
}])

results["source"] = "calculated"
given_values["source"] = "taken from SLUAAR5 Table 2-3 (outside this notebook's reference range)"
all_results = pd.concat([results, given_values], ignore_index=True)
all_results

,barrier,insulation,OVC,impulse_V,working_V_rms,clearance_mm,creepage_mm,source
0,MAINS - MAINS,basic,III,4000,400,3.0,2.00,calculated
1,MAINS - DC_BUS,basic,III,4000,400,3.0,2.00,calculated
2,MAINS - EARTH,basic,III,4000,400,3.0,2.00,calculated
3,MAINS - COLD,reinforced,III,6000,440,5.5,4.40,calculated
4,MAINS - MOTOR,basic,III,4000,440,3.0,2.20,calculated
5,DC_BUS - DC_BUS,functional,I,1500,565,0.5,2.85,calculated
6,DC_BUS - EARTH,basic,III,4000,400,3.0,2.00,calculated
7,DC_BUS - COLD,reinforced,III,6000,440,5.5,4.40,calculated
8,DC_BUS - MOTOR,functional,I,1500,565,0.5,2.85,calculated
9,EARTH - MOTOR,basic,III,4000,400,3.0,2.00,calculated


## 10. Altitude correction

This design's altitude is under 2000 m, so no correction is needed. But
it's worth showing how it works, since it's an easy thing to forget: above
2000 m, every **clearance** value (not creepage -- that's a surface effect,
not an air-gap effect) gets multiplied by the Table A.2 factor.

In [33]:
env_high_alt = Environment(pollution_degree=PollutionDegree.PD2, material_group="IIIa", altitude_m=4000)
calc_high_alt = IEC60664Calculator(env_high_alt)

base_clearance = calc.clearance_mm(4000)  # basic insulation, OVC III
corrected = calc_high_alt.altitude_corrected_clearance_mm(base_clearance)
factor = env_high_alt.altitude_correction_factor()
print(f"Basic-insulation clearance at 2000 m: {base_clearance} mm")
print(f"Same barrier at 4000 m: {base_clearance} mm x {factor} = {corrected} mm")

Basic-insulation clearance at 2000 m: 3.0 mm
Same barrier at 4000 m: 3.0 mm x 1.29 = 3.87 mm


## 11. From barriers to PCB design rules

The last step turns the per-barrier numbers into something you'd actually
type into a PCB tool's constraint manager: a domain-to-domain matrix of
minimum clearance and minimum creepage.

In [34]:
domain_names = [d.name for d in domains]
clearance_matrix = pd.DataFrame(index=domain_names, columns=domain_names, dtype=float)
creepage_matrix = pd.DataFrame(index=domain_names, columns=domain_names, dtype=float)

for _, row in all_results.iterrows():
    a, b = row["barrier"].split(" - ")
    clearance_matrix.loc[a, b] = clearance_matrix.loc[b, a] = row["clearance_mm"]
    creepage_matrix.loc[a, b] = creepage_matrix.loc[b, a] = row["creepage_mm"]

print("Minimum CLEARANCE (mm) between domains:")
display(clearance_matrix)
print()
print("Minimum CREEPAGE (mm) between domains:")
display(creepage_matrix)

Minimum CLEARANCE (mm) between domains:


,MAINS,DC_BUS,EARTH,COLD,MOTOR
MAINS,3.0,3.0,3.0,5.5,3.0
DC_BUS,3.0,0.5,3.0,5.5,0.5
EARTH,3.0,3.0,NaN,0.2,3.0
COLD,5.5,5.5,0.2,NaN,5.5
MOTOR,3.0,0.5,3.0,5.5,0.5



Minimum CREEPAGE (mm) between domains:


,MAINS,DC_BUS,EARTH,COLD,MOTOR
MAINS,2.0,2.00,2.00,4.40,2.20
DC_BUS,2.0,2.85,2.00,4.40,2.85
EARTH,2.0,2.00,NaN,0.04,2.00
COLD,4.4,4.40,0.04,NaN,4.40
MOTOR,2.2,2.85,2.00,4.40,2.00


## 12. Summary and how to extend this

Starting from the same environment/system description as TI's SLUAAR5 white
paper, this notebook:

1. modeled the operating environment and voltage domains as small, reusable
   classes,
2. re-derived the paper's own worked clearance/creepage numbers from the
   underlying IEC 60664-1 formulas (and confirmed they match, with one
   flagged rounding discrepancy),
3. applied that same calculator across every domain pair on the board, and
4. produced a domain-to-domain clearance and creepage matrix that maps
   directly onto PCB constraint-manager rules.

To adapt this for a different design, you'd mainly touch three things:

- **`Environment`** - different pollution degree, PCB material/CTI, or
  installation altitude
- **The domain list and `InsulationBarrier` definitions** - a different
  schematic means different voltage domains, and different insulation
  type / OVC / working-voltage assignments between them (these come from
  circuit analysis, simulation, or measurement -- see the paper's own Steps
  1-4 for how)
- **`IEC60664Calculator._CLEARANCE_POINTS` / `_CREEPAGE_POINTS`** - for
  voltages, OVCs, pollution degrees, or material groups outside what's
  covered here, pull the full tables from IEC 60664-1 Annex F (Tables F.1,
  F.2, F.4) and IEC 61800-5-1

Again: this is a worked *teaching example*, not a qualified design tool.
Insulation coordination on a real product is safety-critical -- get the
complete standard tables and a design review from a qualified engineer
before it goes anywhere near a mains-connected board.

**Reference:** Chen Gao, "Circuit Board Insulation Design According to
IEC60664 for Motor Drive Application," Texas Instruments, SLUAAR5, August
2023.